In [3]:
import subprocess, sys, torch

tv = torch.__version__.split("+")[0]
if torch.version.cuda:
    cu  = "cu" + torch.version.cuda.replace(".", "")
    url = f"https://data.pyg.org/whl/torch-{tv}+{cu}.html"
else:
    url = f"https://data.pyg.org/whl/torch-{tv}+cpu.html"

subprocess.run([sys.executable,"-m","pip","install",
    "pyg_lib","torch_scatter","torch_sparse",
    "torch_cluster","torch_spline_conv","-f",url,"-q"], check=False)
subprocess.run([sys.executable,"-m","pip","install","torch_geometric","-q"])
print("✔ PyG installed")

✔ PyG installed


In [4]:
import os, json, numpy as np, torch
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

FUSION = "/kaggle/input/datasets/jarinproma/pgcn-deap-fusion-preprocessed/preprocessed_fusion"   # <-- your existing fusion dataset

# Collect ONE trial-level ViT embedding per (subject, trial) for real-video subjects
X, subj, val, aro = [], [], [], []
for s in sorted(d for d in os.listdir(FUSION) if d.startswith("s")):
    mp = Path(FUSION)/s/"meta.json"
    if not mp.exists() or not json.load(open(mp)).get("has_real_video", False): continue
    for td in sorted((Path(FUSION)/s).glob("trial_*")):
        wins = sorted(td.glob("win_*.pt"))
        if not wins: continue
        g = torch.load(wins[0], weights_only=False)     # all windows in a trial share one video vector
        X.append(g.video.view(-1).numpy()); subj.append(s)
        y = g.y.view(-1).numpy(); val.append(int(y[0])); aro.append(int(y[1]))
X=np.array(X); subj=np.array(subj); val=np.array(val); aro=np.array(aro)
Xs = StandardScaler().fit_transform(X)
nS = len(set(subj))
print(f"Embeddings: {X.shape} | subjects: {nS} | trials: {len(X)}\n")

# (1) Silhouette — does it cluster by PERSON or by EMOTION?
print(f"Silhouette (subject): {silhouette_score(Xs, subj):+.3f}   <- high => clusters by PERSON")
print(f"Silhouette (valence): {silhouette_score(Xs, val):+.3f}   <- ~0 => NOT by emotion\n")

# (2) Nearest-neighbour same-subject rate
_, idx = NearestNeighbors(n_neighbors=2).fit(Xs).kneighbors(Xs)   # [:,1] = closest OTHER point
same = np.mean([subj[i]==subj[j] for i,j in enumerate(idx[:,1])])
print(f"Nearest-neighbour same-subject rate: {same*100:.1f}%   (chance ~{100/nS:.1f}%)\n")

# (3) Video-only classifiers
subj_id = np.array([sorted(set(subj)).index(s) for s in subj])
def cv5(y):
    accs=[]; 
    for tr,te in StratifiedKFold(5,shuffle=True,random_state=42).split(Xs,y):
        accs.append(accuracy_score(y[te], LogisticRegression(max_iter=2000).fit(Xs[tr],y[tr]).predict(Xs[te])))
    return np.mean(accs)*100, np.std(accs)*100
def loso(y):
    accs=[]
    for tr,te in LeaveOneGroupOut().split(Xs,y,groups=subj):
        if len(set(y[tr]))<2: continue
        accs.append(accuracy_score(y[te], LogisticRegression(max_iter=2000).fit(Xs[tr],y[tr]).predict(Xs[te])))
    return np.mean(accs)*100
print(f"Video-only SUBJECT-ID (5-fold): {cv5(subj_id)[0]:.1f}%   (chance {100/nS:.1f}%)  [sanity: expect ~99%]")
print(f"Video-only VALENCE    (5-fold): {cv5(val)[0]:.1f}% ± {cv5(val)[1]:.1f}%   (chance 50%)")
print(f"Video-only AROUSAL    (5-fold): {cv5(aro)[0]:.1f}% ± {cv5(aro)[1]:.1f}%   (chance 50%)")
print(f"Video-only VALENCE    (LOSO ): {loso(val):.1f}%   (chance 50%)  <- KEY: expect ~chance")
print(f"Video-only AROUSAL    (LOSO ): {loso(aro):.1f}%   (chance 50%)  <- KEY: expect ~chance")

Embeddings: (880, 768) | subjects: 22 | trials: 880

Silhouette (subject): +0.465   <- high => clusters by PERSON
Silhouette (valence): +0.007   <- ~0 => NOT by emotion

Nearest-neighbour same-subject rate: 99.4%   (chance ~4.5%)

Video-only SUBJECT-ID (5-fold): 99.7%   (chance 4.5%)  [sanity: expect ~99%]
Video-only VALENCE    (5-fold): 69.0% ± 3.2%   (chance 50%)
Video-only AROUSAL    (5-fold): 64.7% ± 4.1%   (chance 50%)
Video-only VALENCE    (LOSO ): 49.0%   (chance 50%)  <- KEY: expect ~chance
Video-only AROUSAL    (LOSO ): 54.2%   (chance 50%)  <- KEY: expect ~chance
